In [3]:
import sys
import os

parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from pathlib import Path
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.io import ascii
import matplotlib.pyplot as plt

from dask.distributed import Client
import dask.array
from dask.dataframe.utils import make_meta

from hats import read_hats
from hats.inspection import plot_pixels
from hats_import.catalog.file_readers import CsvReader
from hats_import.margin_cache.margin_cache_arguments import MarginCacheArguments
from hats_import.pipeline import ImportArguments, pipeline_with_client

import lsdb

from catalog_filtering import bandFilterLenient, contains_PM
import hpms_pipeline_test_2 as hpms

print("Imported libraries.")

Imported libraries.


In [2]:
bandList = ['G','R','I','Z','Y']
class_star = None
spread_model = 0.05
magnitude_error = 0.05
check_flags = True
mag = 19
query_string = bandFilterLenient(bandList,classStar=class_star,spreadModel=spread_model,magError=magnitude_error,flag=check_flags,mag=mag)
des_cols = (
    [f'CLASS_STAR_{band}' for band in bandList] + 
    [f'FLAGS_{band}' for band in bandList] + 
    ['RA','DEC','COADD_OBJECT_ID'] + 
    [f'SPREAD_MODEL_{band}' for band in bandList] + 
    [f'WAVG_MAG_PSF_{band}' for band in bandList] + 
    [f'WAVG_MAGERR_PSF_{band}' for band in bandList]
)
max_obj_deviation = 0.2
pm_speed_min = 1000 #units are milliarcseconds per year
pm_speed_max = 10**5
milliarc_degree_conversion = 1/(1000*3600)
print("Defined local vars.")

Defined local vars.


In [9]:
PS1_DIR = Path("../../../../../shared/hats/catalogs/ps1/ps1_otmo")

config_dict = {

    # Catalog Specific Parameters:
    "catalog": lsdb.read_hats(PS1_DIR),
    "id_col_name": "COADD_OBJECT_ID",
    "mag_cols": [f'WAVG_MAG_PSF_{band}' for band in ['I', 'G']],
    "mag_err_cols": [f'WAVG_MAGERR_PSF_{band}' for band in ['I', 'G']],

    # Filtering Specific Parameters:
    "query_string": query_string,
    "xmatch_max_neighbors": 100,
    "max_neighbor_dist": 18,
    "min_neighbors": 3,
    "k": 2,
    "max_obj_deviation": 0.2,

    # Additional Pipeline Parameters:
    "debug_mode": True
}
config = hpms.PipelineConfig_from_dict(config_dict)

print("Defined Config")

Defined Config


In [13]:
CATALOG_DIR = Path("../../../../catalogs")
DES_X_GAIA_NAME = "des_dr2_x_gaia_dr3"
DES_X_GAIA_DIR = CATALOG_DIR / DES_X_GAIA_NAME

des_x_gaia = lsdb.read_hats(DES_X_GAIA_DIR)
des_x_gaia_filt = des_x_gaia.query(f'{pm_speed_max**2} >(pmra_gaia**2 + pmdec_gaia**2) > {pm_speed_min**2}')

with Client():
    df = des_x_gaia_filt.compute()

df_no_dupes = df[~df['source_id_gaia'].duplicated(keep='first')]

gaia_ids = df['source_id_gaia']

#dropping because otherwise produces error when performing .apply below
df_no_dupes = df_no_dupes.drop('source_id_gaia', axis=1)

df_no_dupes

,ra_gaia,dec_gaia,pmra_gaia,pmdec_gaia,phot_g_mean_mag_gaia,phot_bp_mean_mag_gaia,phot_rp_mean_mag_gaia
_healpix_29,,,,,,,
613844718320588,43.195868,1.928423,1400.291765,-515.645438,13.919317,14.903977,12.939081
1153482605725265461,1.383284,-37.367744,5633.438088,-2334.721273,7.682494,8.802319,6.61627
...,...,...,...,...,...,...,...
3263270778643414534,352.563657,-47.61685,-562.651573,-973.694222,15.216914,18.328148,13.734485
3289742749783385503,322.704773,-40.714388,1046.642165,-1396.306284,11.778023,13.375048,10.548469
